In [0]:
%run "/Workspace/Users/dungdq.b22kh019@stu.ptit.edu.vn/community_detection"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/204.6 kB ? eta -:--:--
     ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/204.6 kB 767.1 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 92.2/204.6 kB 1.2 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a4/networkx-3.6.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.1 MB ? eta -:--:--
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/2.1 MB 8.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 0.9/2.1 MB 12.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 2.1/2.1 MB 20.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.6 MB/s eta 0:00:00
  Cre

Tổng số cặp similarity: 3225710


Threshold (97th percentile): 0.7343
Số cạnh sau lọc: 99014


In [0]:
%pip install networkx
import networkx as nx
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Graph: 3256 nodes, 99014 edges


Số communities phát hiện được: 94


Modularity score: 0.6309



Top 10 communities lớn nhất:
community_id
0     633
28    375
9     374
20    277
13    210
14    193
24    191
21    161
66    106
10     97
Name: community_size, dtype: int64


Đã lưu community results


In [0]:
def random_walk_with_restart(G, start_nodes, alpha=0.15, max_iter=100, tol=1e-6):
    nodes = list(G.nodes())
    n = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}
    
    # Ma trận chuyển tiếp (transition matrix) - chuẩn hóa theo weight
    from scipy.sparse import lil_matrix, csr_matrix
    
    W = lil_matrix((n, n), dtype=np.float32)
    for u, v, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        if u in node_idx and v in node_idx:
            W[node_idx[u], node_idx[v]] = w
            W[node_idx[v], node_idx[u]] = w
    
    W = csr_matrix(W)
    
    # Chuẩn hóa hàng (row normalization)
    row_sums = np.array(W.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1  
    D_inv = 1.0 / row_sums
    
    # Vector restart: uniform trên start_nodes
    r = np.zeros(n, dtype=np.float32)
    valid_starts = [node_idx[s] for s in start_nodes if s in node_idx]
    if not valid_starts:
        return {}
    r[valid_starts] = 1.0 / len(valid_starts)
    
    # Power iteration
    p = r.copy()
    for _ in range(max_iter):
        p_new = (1 - alpha) * (W.T.dot(p * D_inv)) + alpha * r
        
        if np.linalg.norm(p_new - p, 1) < tol:
            break
        p = p_new
    
    return {nodes[i]: float(p[i]) for i in range(n)}

In [0]:
print("Tính Gateway scores:")
start_time = datetime.now()

# Nhóm subreddits theo community
community_to_nodes = defaultdict(list)
for sub, comm_id in partition.items():
    community_to_nodes[comm_id].append(sub)

large_communities = {c: nodes for c, nodes in community_to_nodes.items() 
                     if len(nodes) >= 3}

print(f"Số communities >= 3 nodes: {len(large_communities)}")

gateway_results = []

for comm_id, comm_nodes in large_communities.items():
    # Start nodes = tất cả node ngoài community
    outside_nodes = [n for n in G.nodes() if partition.get(n) != comm_id]
    
    if len(outside_nodes) < 10:
        continue

    if len(outside_nodes) > 500:
        outside_nodes = list(np.random.choice(outside_nodes, 500, replace=False))
    
    visit_probs = random_walk_with_restart(G, outside_nodes, alpha=0.15)
    
    for node in comm_nodes:
        if node in visit_probs:
            gateway_results.append({
                "subreddit": node,
                "community_id": comm_id,
                "community_size": len(comm_nodes),
                "gateway_score": visit_probs[node],
                "role": "gateway"
            })

df_gateway = pd.DataFrame(gateway_results)

# Normalize score
df_gateway["gateway_score_normalized"] = df_gateway.groupby("community_id")["gateway_score"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-10)
)

df_top_gateways = df_gateway.sort_values("gateway_score", ascending=False).groupby("community_id").head(3)

print(f"\nGateway tính xong: {(datetime.now()-start_time).seconds}s")
print(f"Tổng gateway candidates: {len(df_gateway)}")

Tính Gateway scores:
Số communities >= 3 nodes: 53

Gateway tính xong: 50s
Tổng gateway candidates: 3174


In [0]:
print("\nTính Bridge scores")

bridge_results = []
community_ids = list(large_communities.keys())

for i, comm_x in enumerate(community_ids[:50]):  # giới hạn 50 community đầu
    nodes_x = large_communities[comm_x]
    
    # RWR xuất phát từ community X
    visit_probs = random_walk_with_restart(G, nodes_x, alpha=0.15)
    
    # Tìm node có score cao nhất NGOÀI community X
    for node, score in visit_probs.items():
        node_community = partition.get(node)
        if node_community != comm_x and score > 0.001:  # threshold nhỏ để lọc noise
            bridge_results.append({
                "subreddit": node,
                "source_community": comm_x,
                "target_community": node_community,
                "bridge_score": score,
                "role": "bridge"
            })

df_bridge = pd.DataFrame(bridge_results) if bridge_results else pd.DataFrame()

if not df_bridge.empty:
    df_bridge = df_bridge.sort_values("bridge_score", ascending=False)
    print(f"Tổng bridge candidates: {len(df_bridge)}")
    print("\nTop 10 bridges:")
    print(df_bridge.head(10)[["subreddit", "source_community", "target_community", "bridge_score"]])


Tính Bridge scores
Tổng bridge candidates: 806

Top 10 bridges:
               subreddit  source_community  target_community  bridge_score
729     ConservativeKiwi                47                 0      0.067406
803              ChatGPT                58                 0      0.064398
536        2islamist4you                31                 0      0.057691
680      AEWFightForever                46                28      0.043781
361           Aliexpress                19                 0      0.033455
360  BehindTheClosetDoor                19                 0      0.033399
682                  ANW                46                 0      0.031048
312        CatholicMemes                15                 0      0.029758
758     CharacterAi_NSFW                58                 0      0.022015
799              ChaiApp                58                 0      0.020693


In [0]:
print("Phân tích overlap bridge và gateway")

if not df_bridge.empty:
    top_bridges = set(df_bridge.nlargest(100, "bridge_score")["subreddit"].tolist())

    top_gateways = set(df_top_gateways["subreddit"].tolist())
    
    overlap = top_bridges.intersection(top_gateways)
    overlap_pct = len(overlap) / len(top_bridges) * 100 if top_bridges else 0
    
    print(f"Top 100 bridges: {len(top_bridges)}")
    print(f"Top gateways: {len(top_gateways)}")
    print(f"Overlap: {len(overlap)} nodes ({overlap_pct:.1f}%)")
    print(f"Số lượng {overlap_pct:.1f}% bridges cũng là gateways")

Phân tích overlap bridge và gateway
Top 100 bridges: 95
Top gateways: 159
Overlap: 4 nodes (4.2%)
Số lượng 4.2% bridges cũng là gateways


In [0]:
df_gateway_spark = spark.createDataFrame(df_gateway)
df_gateway_spark.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"wasbs://{container}@{storage_account}.blob.core.windows.net/gateway_results"
)

if not df_bridge.empty:
    df_bridge_spark = spark.createDataFrame(df_bridge)
    df_bridge_spark.coalesce(1).write.mode("overwrite").option("header", True).csv(
        f"wasbs://{container}@{storage_account}.blob.core.windows.net/bridge_results"
    )

print("Đã lưu kết quả gateway_bridge")

Đã lưu kết quả gateway_bridge
